In [13]:
# Simulasi RBAC sederhana
roles_permissions = {
 'Data Owner': ['read', 'write', 'delete', 'grant_access'],
 'Data Steward': ['read', 'write'],
 'Data Analyst': ['read'],
 'Guest': [],
}

def cek_akses(role, aksi):
 izin = roles_permissions.get(role, [])
 if aksi in izin:
    return f"AKSES DIIZINKAN: '{role}' dapat melakukan '{aksi}'"
 else:
    return f"AKSES DITOLAK: '{role}' TIDAK dapat melakukan '{aksi}'"

# Uji simulasi RBAC
print(cek_akses('Data Analyst', 'read'))
print(cek_akses('Data Analyst', 'delete'))
print(cek_akses('Data Owner', 'delete'))


AKSES DIIZINKAN: 'Data Analyst' dapat melakukan 'read'
AKSES DITOLAK: 'Data Analyst' TIDAK dapat melakukan 'delete'
AKSES DIIZINKAN: 'Data Owner' dapat melakukan 'delete'


In [14]:
import datetime

audit_log = []

def catat_aktivitas(user, role, aksi, status):
 log_entry = {
  'timestamp': datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
  'user': user,
  'role': role,
  'aksi': aksi,
  'status': status
 }

 audit_log.append(log_entry)
 return log_entry

# Simulasikan beberapa aktivitas pengguna
catat_aktivitas('budi01', 'Data Analyst', 'read dataset Titanic', 'SUCCESS')
print(cek_akses('Guest', 'read'))
catat_aktivitas('budi01', 'Data Analyst', 'delete dataset Titanic', 'DENIED')
catat_aktivitas('admin01', 'Data Owner', 'grant_access ke citra02', 'SUCCESS')

# Tampilkan audit log dalam bentuk tabel
import pandas as pd

df_audit = pd.DataFrame(audit_log)

print(df_audit)

AKSES DITOLAK: 'Guest' TIDAK dapat melakukan 'read'
             timestamp     user          role                     aksi  \
0  2026-10-25 03:56:52   budi01  Data Analyst     read dataset Titanic   
1  2026-10-25 03:56:52   budi01  Data Analyst   delete dataset Titanic   
2  2026-10-25 03:56:52  admin01    Data Owner  grant_access ke citra02   

    status  
0  SUCCESS  
1   DENIED  
2  SUCCESS  


In [15]:
import hashlib
import pandas as pd
import seaborn as sns
df = sns.load_dataset('titanic').reset_index()

# PSEUDONYMIZATION: mengganti index dengan kode hash yang dapat dipetakan kembali
def buat_pseudonim(id_asli):
 return hashlib.sha256(str(id_asli).encode()).hexdigest()[:10]
df['id_pseudonim'] = df['index'].apply(buat_pseudonim)

print(df[['index', 'id_pseudonim']].head())

# Simpan tabel pemetaan (mapping table) secara terpisah dan aman
mapping_table = df[['index', 'id_pseudonim']].copy()

print('\nTabel pemetaan (harus disimpan terpisah dan diakses terbatas):')
print(mapping_table.head())

   index id_pseudonim
0      0   5feceb66ff
1      1   6b86b273ff
2      2   d4735e3a26
3      3   4e07408562
4      4   4b227777d4

Tabel pemetaan (harus disimpan terpisah dan diakses terbatas):
   index id_pseudonim
0      0   5feceb66ff
1      1   6b86b273ff
2      2   d4735e3a26
3      3   4e07408562
4      4   4b227777d4


In [16]:
# ANONYMIZATION: menghapus/menggeneralisasi kolom yang berpotensi identifikasi individu
df_anon = df.copy()

# Hapus kolom identifikasi langsung (contoh: index asli, nama jika ada)
df_anon = df_anon.drop(columns=['index', 'id_pseudonim'], errors='ignore')

# Generalisasi usia menjadi rentang (mengurangi risiko re-identifikasi)
def generalisasi_usia(usia):
 if pd.isna(usia):
  return 'Tidak diketahui'
 elif usia < 20:
  return '0-19'
 elif usia < 40:
  return '20-39'
 elif usia < 60:
  return '40-59'
 else:
  return '60+'

df_anon['rentang_usia'] = df_anon['age'].apply(generalisasi_usia)
df_anon = df_anon.drop(columns=['age'])

print('Data setelah anonymization (kolom age digeneralisasi, id dihapus):')
print(df_anon[['rentang_usia', 'sex', 'pclass']].head())


Data setelah anonymization (kolom age digeneralisasi, id dihapus):
  rentang_usia     sex  pclass
0        20-39    male       3
1        20-39  female       1
2        20-39  female       3
3        20-39  female       1
4        20-39    male       3


In [17]:
# Ringkasan hasil audit
print('Ringkasan status akses:')
print(df_audit['status'].value_counts())
print('\nRingkasan aktivitas per role:')
print(df_audit.groupby('role')['aksi'].count())

# Tambahan: aksi yang paling sering ditolak
print('Aksi yang ditolak (DENIED):')
print(df_audit[df_audit['status']=='DENIED']['aksi'].value_counts())

Ringkasan status akses:
status
SUCCESS    2
DENIED     1
Name: count, dtype: int64

Ringkasan aktivitas per role:
role
Data Analyst    2
Data Owner      1
Name: aksi, dtype: int64
Aksi yang ditolak (DENIED):
aksi
delete dataset Titanic    1
Name: count, dtype: int64


In [18]:
import json, os
import pandas as pd

# Langkah 06 - data/kolom berisiko (privacy issues)
privacy_summary = pd.DataFrame([
    ['index (ID baris), nama*', 'Pengenal langsung',          'Tinggi', 'Menunjuk satu individu secara langsung'],
    ['age',                     'Quasi-identifier',           'Sedang', 'Usia tepat mempersempit identitas bila digabung atribut lain'],
    ['deck',                    'Quasi-identifier',           'Sedang', 'Hanya 203 dari 891 baris terisi sehingga mudah dikenali'],
    ['fare',                    'Quasi-identifier/keuangan',  'Sedang', '248 nilai unik; informasi pengeluaran pribadi'],
    ['survived, alive',         'Atribut sensitif (status)',  'Sedang', 'Informasi nasib individu'],
    ['embarked, embark_town',   'Quasi-identifier',           'Rendah', 'Hanya 3 kategori'],
    ['sex, pclass, class, who', 'Quasi-identifier',           'Rendah', 'Kategori umum'],
    ['sibsp, parch, alone',     'Informasi keluarga',         'Rendah', 'Berisiko bila dikombinasikan'],
], columns=['kolom', 'kategori', 'risiko_jika_bocor', 'alasan'])

# Langkah 07 - Risk Assessment Matrix (skor = Likelihood + Impact)
skala = {'Rendah': 1, 'Sedang': 2, 'Tinggi': 3}
def risk_level(l, i):
    s = skala[l] + skala[i]
    return 'Rendah' if s == 2 else ('Sedang' if s <= 4 else 'Tinggi')

mit_df = pd.DataFrame([
    ['R1',  'Data identitas penumpang',            'Sedang', 'Tinggi', 'Pseudonymization + akses terbatas (RBAC)'],
    ['R2',  'Tabel pemetaan pseudonim',            'Rendah', 'Tinggi', 'Simpan terpisah, enkripsi, hanya Data Owner'],
    ['R3',  'Kombinasi quasi-identifier',          'Tinggi', 'Sedang', 'Generalisasi tarif/kabin, uji k-anonymity (k>=5)'],
    ['R4',  'Status keselamatan (survived, alive)','Sedang', 'Sedang', 'Akses sesuai peran, tampilkan agregat'],
    ['R5',  'Data usia',                           'Rendah', 'Sedang', 'Anonymization (rentang usia)'],
    ['R6',  'Data kabin dan pelabuhan',            'Rendah', 'Sedang', 'Generalisasi atau hapus bila tidak perlu'],
    ['R7',  'Data keluarga',                       'Rendah', 'Rendah', 'Tidak perlu penanganan khusus'],
    ['R8',  'Data kelas tiket dan tarif',          'Rendah', 'Rendah', 'Tidak perlu penanganan khusus'],
    ['R9',  'Audit log',                           'Rendah', 'Sedang', 'Log append-only, akses terbatas'],
    ['R10', 'File model-ready',                    'Sedang', 'Sedang', 'Read-only, hapus kolom tidak dipakai, enkripsi'],
], columns=['id', 'kelompok_data', 'likelihood', 'impact', 'mitigasi'])
mit_df['risk_level'] = [risk_level(l, i) for l, i in zip(mit_df['likelihood'], mit_df['impact'])]

audit_df = pd.DataFrame(audit_log)
report = {
    'audit_log':      audit_df.to_dict(orient='records'),
    'privacy_issues': privacy_summary.to_dict(orient='records'),
    'mitigations':    mit_df.to_dict(orient='records'),
}
with open('audit_report.json', 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

audit_df.to_csv('audit_log.csv', index=False)
privacy_summary.to_csv('privacy_issues.csv', index=False)
mit_df.to_csv('mitigations.csv', index=False)
mapping_table.to_csv('mapping_table.csv', index=False)
df_anon.to_csv('titanic_anonymized.csv', index=False)

print("Audit report saved: audit_report.json, audit_log.csv, privacy_issues.csv, mitigations.csv, mapping_table.csv, titanic_anonymized.csv")
print('\nFile hasil di folder kerja:', sorted(f for f in os.listdir('.') if f.endswith(('.csv', '.json'))))

Audit report saved: audit_report.json, audit_log.csv, privacy_issues.csv, mitigations.csv, mapping_table.csv, titanic_anonymized.csv

File hasil di folder kerja: ['audit_log.csv', 'audit_report.json', 'mapping_table.csv', 'mitigations.csv', 'privacy_issues.csv', 'titanic_anonymized.csv', 'titanic_model_ready.csv']


PRAKTIKUM 2

In [19]:
from datetime import datetime
roles = {
 'Purser':            {'read':['name','age','sex','pclass','ticket','fare'], 'write':['ticket']},
 'Data_Scientist':    {'read':['age','sex','pclass','fare','survived'],      'write':[]},
 'Compliance_Officer':{'read':['*'],                                         'write':[]},
 'Customer_Service':  {'read':['name','ticket','pclass'],                    'write':['ticket']},
 'Marketing':         {'read':['pclass','age'],                              'write':[]},
}

audit_log = []
def access(user, role, action, column):
    perm = roles[role].get(action, [])
    ok = ('*' in perm) or (column in perm)
    audit_log.append({'time':datetime.now().strftime('%H:%M:%S'),
        'user':user,'role':role,'action':action,'column':column,
        'result':'GRANTED' if ok else 'DENIED'})
    return ok

# 6 skenario
access('Andi','Data_Scientist','read','age')
access('Andi','Data_Scientist','read','name')          # ditolak
access('Budi','Marketing','read','fare')               # ditolak
access('Sari','Compliance_Officer','read','name')
access('Rina','Customer_Service','write','ticket')
access('Rina','Customer_Service','write','fare')       # ditolak
pd.DataFrame(audit_log)

,time,user,role,action,column,result
0,03:56:53,Andi,Data_Scientist,read,age,GRANTED
1,03:56:53,Andi,Data_Scientist,read,name,DENIED
2,03:56:53,Budi,Marketing,read,fare,DENIED
3,03:56:53,Sari,Compliance_Officer,read,name,GRANTED
4,03:56:53,Rina,Customer_Service,write,ticket,GRANTED
5,03:56:53,Rina,Customer_Service,write,fare,DENIED


In [20]:
import hashlib
raw = sns.load_dataset('titanic').copy()
raw['name'] = ['Penumpang_'+str(i) for i in raw.index]

SALT = 'rahasia_perusahaan'
pseudo = raw.copy()
pseudo['name'] = pseudo['name'].apply(lambda x: hashlib.sha256((x+SALT).encode()).hexdigest()[:10])

anon = raw.drop(columns=['name']).copy()
anon['age'] = pd.cut(anon.age, [0,12,18,40,60,100], labels=['<12','12-18','19-40','41-60','>60'])
anon['fare'] = pd.qcut(anon.fare, 4, labels=['Q1','Q2','Q3','Q4'])
print(pseudo[['name']].head()); print(anon[['age','fare']].head())

         name
0  6e23c4cf3b
1  ddeb2c22c1
2  be11d89368
3  6522091819
4  865eb6d9e2
     age fare
0  19-40   Q1
1  19-40   Q4
2  19-40   Q2
3  19-40   Q4
4  19-40   Q2
